# 02 — AAA section-aware chunking and local embeddings

Builds RAG chunks from the cleaned pages produced by notebook 01, validates chunk quality, then embeds only valid chunks with a local free model.

No paid embedding API is used. Chunk text is source text, not a paraphrase.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd.parent]:
    if (candidate / "ingestion" / "chunking.py").exists():
        sys.path.insert(0, str(candidate))
        break

import ingestion.chunking as cc
import ingestion.preprocess as cp
import retrieval.index as cr

PROJECT_ROOT = cp.find_project_root()
print("Project root:", PROJECT_ROOT)
print("Embedding model:", cr.DEFAULT_MODEL)


Project root: C:\Users\DELL\Downloads\aaa-clinical-rag
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## Chunking

Chunks prefer section boundaries, keep recommendations with surrounding context, and avoid splitting numeric ranges.


In [2]:
chunk_result = cc.run_chunking(PROJECT_ROOT)
cc.print_chunk_report(chunk_result["quality"])
chunks = chunk_result["chunks"]
print("\nWrote", chunk_result["relative_path"])
print("Chunks by document:")
display(pd.DataFrame(chunks).groupby(["document_id", "document_name"]).size().rename("n_chunks").to_frame())


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1547 > 512). Running this sequence through the model will result in indexing errors


CHUNKS
- total: 2116
- valid: 2116
- invalid: 0
- duplicates: 2
- missing metadata issues: 0
- missing section_title: 530

TOKENS (tokenizer: sentence-transformers/all-MiniLM-L6-v2, limit 256)
- chunks: 2116
- min: 12
- median: 206.0
- mean: 194.78
- max: 254
- EXCEEDING_LIMIT = 0

CONTENT TYPES
- boilerplate: 14
- clinical: 1330
- reference: 661
- title_only: 8
- toc: 103

- status: PASS

Wrote data/chunks/chunks.json
Chunks by document:


,,n_chunks
document_id,document_name,
ESVS_2024,Editor's Choice -- European Society for Vascular Surgery (ESVS) 2024 Clinical Practice Guidelines on the Management of Abdominal Aorto-Iliac Artery Aneurysms&#x2606;,1806
NICE_NG156,Abdominal aortic aneurysm: diagnosis and management,158
SVS_2018,Care of Patients with an Abdominal Aortic Aneurysm,48
USPSTF_2019,Screening for Abdominal Aortic Aneurysm: US Preventive Services Task Force Recommendation Statement,104


In [3]:
chunk_df = pd.DataFrame(chunks)
print("Chunk metadata sample")
display(chunk_df[["chunk_id", "document_id", "section_title", "page_start", "page_end", "content_type", "token_count", "char_count"]].head(12))

print("\nToken distribution (tokenizer of the embedding model)")
stats = chunk_result["quality"]["tokens"]
display(pd.Series(stats).to_frame("value"))

print("\nChunks by content type (low-value types are kept on disk but excluded from the index)")
display(chunk_df["content_type"].value_counts().rename("n_chunks").to_frame())

print("\nSample chunks - metadata AND the real chunk text")
cc.print_chunk_samples(chunks, per_document=2, max_chars=700)

# Hard gate: nothing may exceed the embedding model's window, because
# SentenceTransformer.encode() would silently truncate it.
assert stats["exceeding_limit"] == 0, f"{stats['exceeding_limit']} chunk(s) exceed {stats['token_limit']} tokens"
assert chunk_result["quality"]["status"] == "PASS", chunk_result["quality"]["errors"]
assert (PROJECT_ROOT / "data" / "chunks" / "chunks.json").stat().st_size > 0
print(f"\nEXCEEDING_LIMIT = {stats['exceeding_limit']}")


Chunk metadata sample


,chunk_id,document_id,section_title,page_start,page_end,content_type,token_count,char_count
0,ESVS_2024__p1-1__c0001,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,220,696
1,ESVS_2024__p1-1__c0002,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,206,754
2,ESVS_2024__p1-1__c0003,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,216,1119
3,ESVS_2024__p1-1__c0004,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,209,979
4,ESVS_2024__p1-1__c0005,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,220,966
5,ESVS_2024__p1-1__c0006,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,217,861
6,ESVS_2024__p1-1__c0007,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,205,698
7,ESVS_2024__p1-1__c0008,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,199,642
8,ESVS_2024__p1-1__c0009,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,220,728
9,ESVS_2024__p1-1__c0010,ESVS_2024,CLINICAL PRACTICE GUIDELINE DOCUMENT,1,1,clinical,215,719



Token distribution (tokenizer of the embedding model)


,value
model_name,sentence-transformers/all-MiniLM-L6-v2
token_limit,256
n_chunks,2116
min_tokens,12
max_tokens,254
mean_tokens,194.78
median_tokens,206.0
exceeding_limit,0



Chunks by content type (low-value types are kept on disk but excluded from the index)


,n_chunks
content_type,
clinical,1330
reference,661
toc,103
boilerplate,14
title_only,8



Sample chunks - metadata AND the real chunk text
----------------------------------------------------------------------------------------
chunk_id     : ESVS_2024__p1-1__c0001
document     : Editor's Choice -- European Society for Vascular Surgery (ESVS) 2024 Clinical Practice Guidelines on the Management of Abdominal Aorto-Iliac Artery Aneurysms&#x2606; (ESVS_2024)
is_guideline : True
page         : 1 (pages 1-1)
section      : CLINICAL PRACTICE GUIDELINE DOCUMENT [detected]
content_type : clinical
tokens       : 220   chars: 696
text         :
CLINICAL PRACTICE GUIDELINE DOCUMENT
Editor’s Choice – European Society for Vascular Surgery (ESVS) 2024 Clinical
Practice Guidelines on the Management of Abdominal Aorto-Iliac Artery
Aneurysms5
Anders Wanhainen a,*, Isabelle Van Herzeele a, Frederico Bastos Goncalves a, Sergi Bellmunt Montoya a, Xavier Berard a, Jonathan R. Boyle a,
Mario D’Oria a, Carlota F. Prendes a, Christos D. Karkos a, Arkadiusz Kazimierczak a, Mark J.W. Koelemay a, Til

## Local embeddings and vector index

Model: `sentence-transformers/all-MiniLM-L6-v2` (configurable in `clinical_rag.DEFAULT_MODEL`).
Index: cosine similarity over L2-normalized NumPy vectors, with chunk metadata stored beside the matrix.


In [4]:
embed_result = cr.run_embedding_index(PROJECT_ROOT, model_name=cr.DEFAULT_MODEL)
print("Embedded chunks:", embed_result["total_embedded"])
print("Model:", embed_result["model_name"])
print("Dimension:", embed_result["embedding_dim"])
print("Failed embeddings:", embed_result["failed"])
print("Index files:")
for key, path in embed_result["paths"].items():
    print(f" - {key}: {path}")

index_dir = PROJECT_ROOT / "data" / "embeddings"
meta = json.loads((index_dir / "index_meta.json").read_text(encoding="utf-8"))
assert meta["n_vectors"] == embed_result["total_embedded"]
assert (index_dir / "embeddings.npy").stat().st_size > 0
print("\nNotebook 02 embeddings validated.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4430.86it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Batches:   2%|▏         | 1/42 [00:02<01:40,  2.45s/it]

Batches:   5%|▍         | 2/42 [00:04<01:18,  1.95s/it]

...

Batches:  95%|█████████▌| 40/42 [01:04<00:03,  1.64s/it]

Batches:  98%|█████████▊| 41/42 [01:06<00:01,  1.63s/it]

Batches: 100%|██████████| 42/42 [01:07<00:00,  1.30s/it]

Batches: 100%|██████████| 42/42 [01:07<00:00,  1.60s/it]

Embedded chunks: 1330
Model: sentence-transformers/all-MiniLM-L6-v2
Dimension: 384
Failed embeddings: []
Index files:
 - dir: data/embeddings
 - vectors: data/embeddings/embeddings.npy
 - meta: data/embeddings/index_meta.json
 - chunks: data/embeddings/embedded_chunks.json

Notebook 02 embeddings validated.
